# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [1]:
# Example data from article about DMACN

from scipy.io import loadmat
import torch

# Load the data
PTSD = loadmat("C:\\Users\\oddar\\Downloads\\PTSD_connectivity.mat")
# PTSD is a dataset containing 87 samples (subjects) with 340 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 3

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = PTSD["connectivities"]  # Example matrix

PTSD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = PTSD_tensor  # [N,d] float tensor
print("Data shape:", X.shape)

Data shape: torch.Size([87, 340])


In [1]:
# Loading the data with 29 features (for 29x29 data)
# subject_features.npz contains the vectorized upper triangle of the 29x29 FC matrices, resulting in 50 features per sample.
from scipy.io import loadmat
import torch
from data_loader import load_workable_fc


filepath = "Prosjektoppgave-Odd-Arne-og-Mats-main\\subject_features.npz" # All subjects, 50 features

Functional_connectivity_matrix = load_workable_fc(filepath)

Functional_connectivity_matrix.to_csv("subject_features_clean.csv", index=True)

print(Functional_connectivity_matrix.shape)

X = torch.from_numpy(Functional_connectivity_matrix.values).float()


(141, 50)
(141, 29)
(141, 29)


In [2]:
# Loading the data with all 200 features (for 200x200 data)
# 200_schaefer_vectorized_fc.mat contains the vectorized upper triangle of the 200x200 FC matrices, resulting in 19900 features per sample.

from scipy.io import loadmat
import torch
import numpy as np

FC_test_mat = loadmat("C:\\Users\\oddar\\Downloads\\200_schaefer_vectorized_fc.mat")
# FC_test_mat = loadmat("C:\\Mats og Odd Arne\\Prosjektoppgave\\sch407\\YA\\200_schaefer_vectorized_fc.mat")  # Load the .mat file

FC_test_array = FC_test_mat["200_vectorized_fc"]  # Example matrix
# np.fill_diagonal(FC_test_array, 1.0)  # Set diagonal to zero

print(FC_test_array[0:5].shape)  # Print the first 5 rows to verify

X = torch.from_numpy(FC_test_array).float()  # Example matrix

# Check if any values are abs(X) > 1.0
if torch.any(torch.abs(X) > 1.0):
    print("Warning: Some values in X have absolute value greater than 1.0, which may cause numerical issues in the polynomial kernel.")

(5, 19900)


Define the autoencoder specs

In [3]:
kernel_specs = [
    {"kind": "rbf", "t0": 0.01},
    {"kind": "rbf", "t0": 0.05},
    {"kind": "rbf", "t0": 0.1},
    {"kind": "rbf", "t0": 1},
    {"kind": "rbf", "t0": 10},
    {"kind": "rbf", "t0": 50},
    {"kind": "rbf", "t0": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3

In [3]:
# Config for 340x340 data
from DMACN import DMACN, DMACNConfig


cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[340, 285, 240, 202, 170],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[170, 202, 240, 285, 340],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

In [ ]:
from DMACN import DMACN, DMACNConfig


# Config for 29x29 data
cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[29, 25, 22, 19, 17],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[17, 19, 22, 25, 29],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)


In [4]:
from DMACN import DMACN, DMACNConfig

# Config for 19900x19900 data
cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[19900, 15795, 12536, 9950],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[9950, 12536, 15795, 19900],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=100,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=200,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)


Run the model

In [5]:
model = DMACN(cfg)
model.fit(X, verbose_every=1)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

Ready to train DMACN:
epoch    1/200 [mid-only]  J=7.3844e+03  J1=1.7474e+02  J2=9.4496e-02  J3=7.2096e+03  omega_sum=1.0000
epoch    2/200 [multilayer]  J=7.3772e+03  J1=1.7473e+02  J2=3.0213e-02  J3=7.2024e+03  omega_sum=1.0000
epoch    3/200 [multilayer]  J=7.3699e+03  J1=1.7472e+02  J2=3.0231e-02  J3=7.1952e+03  omega_sum=1.0000
epoch    4/200 [multilayer]  J=7.3627e+03  J1=1.7471e+02  J2=3.0211e-02  J3=7.1880e+03  omega_sum=1.0000
epoch    5/200 [multilayer]  J=7.3555e+03  J1=1.7470e+02  J2=3.0251e-02  J3=7.1808e+03  omega_sum=1.0000
epoch    6/200 [multilayer]  J=7.3484e+03  J1=1.7469e+02  J2=3.0177e-02  J3=7.1736e+03  omega_sum=1.0000
epoch    7/200 [multilayer]  J=7.3412e+03  J1=1.7468e+02  J2=3.0205e-02  J3=7.1665e+03  omega_sum=1.0000
epoch    8/200 [multilayer]  J=7.3340e+03  J1=1.7467e+02  J2=3.0171e-02  J3=7.1593e+03  omega_sum=1.0000
epoch    9/200 [multilayer]  J=7.3268e+03  J1=1.7466e+02  J2=3.0231e-02  J3=7.1521e+03  omega_sum=1.0000
epoch   10/200 [multilayer]  J=7.31

In [5]:
# Evaluate PTSD clustering performance using PTSD_clinical_labels.csv
import pandas as pd
clinical_labels = pd.read_csv("PTSD_clinical.scv.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (87)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)

Adjusted Rand Index (ARI): 1.0


### UMAP

In [ ]:
from UMAP import UMAP

C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ImportError: cannot import name 'UMAP' from 'UMAP' (c:\Users\oddar\OneDrive - NTNU\Documents\Universitet\Masteroppgåve\Master-Thesis-Autoencoder-and-UMAP-clustering-on-functional-connectivity\UMAP.py)

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [6]:
from Evaluate_models import evaluate_clustering, evaluate_single_clustering
import os

# Define where to find the labels 
labels_path = "Clusters\DMACN__Clusters_3__label_0_21_label_1_19_label_2_32.txt"
evaluate_single_clustering(functional_connectivity_matrix=X, labels_path=labels_path)


 Scores for the given labels:
Silhouette coefficient: 0.002338581718504429
Davies-Bouldin score: 5.71068601622989
Calinski-Harabasz score: 1.7393631346468488
